In [ ]:
!pip install transformers datasets torch accelerate -q


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'leetcode_dataset - lc.csv', '.ipynb_checkpoints', 'sample_data']


In [ ]:
import pandas as pd

# Load with correct filename
df = pd.read_csv('leetcode_dataset - lc.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst row:")
print(df.iloc[0])
print("\nDifficulty counts:")
print(df['difficulty'].value_counts())
print("\nPremium counts:")
print(df['is_premium'].value_counts())

Shape: (1825, 19)

Columns: ['id', 'title', 'description', 'is_premium', 'difficulty', 'solution_link', 'acceptance_rate', 'frequency', 'url', 'discuss_count', 'accepted', 'submissions', 'companies', 'related_topics', 'likes', 'dislikes', 'rating', 'asked_by_faang', 'similar_questions']

First row:
id                                                                   1
title                                                          Two Sum
description          Given an array of integers `nums` and an integ...
is_premium                                                           0
difficulty                                                        Easy
solution_link                                        /articles/two-sum
acceptance_rate                                                   46.7
frequency                                                        100.0
url                              https://leetcode.com/problems/two-sum
discuss_count                                                

In [ ]:
import pandas as pd
import json
import re

# Filter: only free problems, remove nulls
df_clean = df[df['is_premium'] == 0].copy()
df_clean = df_clean.dropna(subset=['title', 'description', 'difficulty'])
df_clean = df_clean[df_clean['description'].str.len() > 100]

# Clean description — remove HTML tags and markdown backticks
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML
    text = re.sub(r'`+', '', text)             # remove backticks
    text = re.sub(r'\*+', '', text)            # remove asterisks
    text = re.sub(r'\n+', ' ', text)           # flatten newlines
    text = re.sub(r'\s+', ' ', text)           # collapse spaces
    return text.strip()

df_clean['description'] = df_clean['description'].apply(clean_text)
df_clean['title']       = df_clean['title'].apply(clean_text)

# Map difficulty to BuildIMS levels
level_map = { 'Easy': 'beginner', 'Medium': 'intermediate', 'Hard': 'advanced' }
df_clean['level'] = df_clean['difficulty'].map(level_map)

# Map related_topics to day_type
def get_day_type(topics):
    topics = str(topics).lower()
    if any(x in topics for x in ['design', 'oop', 'system', 'architecture']):
        return 'design'
    elif any(x in topics for x in ['debug', 'error', 'fix', 'bug']):
        return 'debug'
    elif any(x in topics for x in ['dynamic', 'recursion', 'math', 'bit']):
        return 'explain'
    elif any(x in topics for x in ['array', 'string', 'hash', 'tree', 'graph', 'linked']):
        return 'build'
    else:
        return 'code'

df_clean['day_type'] = df_clean['related_topics'].apply(get_day_type)

# Keep only what we need
df_final = df_clean[['title', 'description', 'level', 'day_type', 'url']].reset_index(drop=True)

print(f"Clean dataset size: {len(df_final)}")
print(f"\nLevel distribution:\n{df_final['level'].value_counts()}")
print(f"\nDay type distribution:\n{df_final['day_type'].value_counts()}")
print(f"\nSample row:")
print(df_final.iloc[0])

Clean dataset size: 1408

Level distribution:
level
intermediate    740
beginner        358
advanced        310
Name: count, dtype: int64

Day type distribution:
day_type
build      592
explain    431
code       344
design      38
debug        3
Name: count, dtype: int64

Sample row:
title                                                    Two Sum
description    Given an array of integers nums and an integer...
level                                                   beginner
day_type                                                   build
url                        https://leetcode.com/problems/two-sum
Name: 0, dtype: object


In [ ]:
# Format each problem as a training text block
# GPT-2 learns the pattern: given this format, generate similar challenges

def format_challenge(row):
    return f"""<|challenge|>
TITLE: {row['title']}
TYPE: {row['day_type']}
LEVEL: {row['level']}
DESCRIPTION: {row['description'][:600]}
<|endchallenge|>"""

# Create training texts
texts = df_final.apply(format_challenge, axis=1).tolist()

# Save to text file for training
with open('challenges_train.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(texts))

print(f"Total training examples: {len(texts)}")
print(f"\nSample training text:")
print(texts[0])
print("\n---")
print(texts[5])

Total training examples: 1408

Sample training text:
<|challenge|>
TITLE: Two Sum
TYPE: build
LEVEL: beginner
DESCRIPTION: Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target. You may assume that each input would have exactly one solution, and you may not use the same element twice. You can return the answer in any order. Example 1: Input: nums = [2,7,11,15], target = 9 Output: [0,1] Output: Because nums[0] + nums[1] == 9, we return [0, 1]. Example 2: Input: nums = [3,2,4], target = 6 Output: [1,2] Example 3: Input: nums = [3,3], target = 6 Output: [0,1] Constraints: 2 <= nums.length <= 103 -109 <= nums[i] <= 109 -109 <= target <= 109 Only one va
<|endchallenge|>

---
<|challenge|>
TITLE: ZigZag Conversion
TYPE: build
LEVEL: intermediate
DESCRIPTION: The string "PAYPALISHIRING" is written in a zigzag pattern on a given number of rows like this: (you may want to display this pattern in a fixed font for better legibilit

In [ ]:
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
import torch

class ChallengeDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=256):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        tokenized = tokenizer(text, return_tensors='pt', truncation=False)['input_ids'][0]
        self.examples = []
        for i in range(0, len(tokenized) - block_size, block_size):
            self.examples.append(tokenized[i:i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {'input_ids': self.examples[idx], 'labels': self.examples[idx]}

print("Loading tokenizer and model...")
model_name = "distilgpt2"
tokenizer  = GPT2Tokenizer.from_pretrained(model_name)
model      = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({
    'pad_token': '<|pad|>',
    'additional_special_tokens': ['<|challenge|>', '<|endchallenge|>']
})
model.resize_token_embeddings(len(tokenizer))

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Model parameters:     {model.num_parameters():,}")

dataset       = ChallengeDataset(tokenizer, 'challenges_train.txt', block_size=256)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Dataset size: {len(dataset)} blocks")

training_args = TrainingArguments(
    output_dir='./challenge_model',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=1,
    logging_steps=50,
    learning_rate=5e-4,
    warmup_steps=100,
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

print("\nStarting training...")
trainer.train()
print("\nTraining complete!")

trainer.save_model('./challenge_model')
tokenizer.save_pretrained('./challenge_model')
print("Model saved to ./challenge_model")

Loading tokenizer and model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer vocab size: 50260
Model parameters:     81,914,880


Token indices sequence length is longer than the specified maximum sequence length for this model (259394 > 1024). Running this sequence through the model will result in indexing errors


...

Step,Training Loss
50,3.018148
100,2.374015
150,2.188132
200,2.015027
250,1.934799
300,1.580801
350,1.627504
400,1.508904
450,1.331906
500,1.351219


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./challenge_model


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

print("Loading trained model...")
tokenizer = GPT2Tokenizer.from_pretrained('./challenge_model')
model     = GPT2LMHeadModel.from_pretrained('./challenge_model')
model.eval()

def generate_challenge(day_type='code', level='intermediate'):
    prompt = f"<|challenge|>\nTITLE:"
    inputs = tokenizer.encode(prompt, return_tensors='pt')

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=200,
            temperature=0.85,
            top_p=0.92,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode('<|endchallenge|>')[0],
            repetition_penalty=1.2,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print("\n--- Generated Challenge ---")
    print(text)
    print("----------------------------\n")
    return text

# Test 3 generations
generate_challenge('code',    'beginner')
generate_challenge('build',   'intermediate')
generate_challenge('explain', 'advanced')

Loading trained model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


--- Generated Challenge ---
<|challenge|>
TITLE: Construct Binary Tree from Preorder Traversal
TYPE: build
LEVEL: intermediate
DESCRIPTION: You are given a pre-order traversal of a binary tree. Return the prerequisites for this traversal. A pre-order traversal is an initialization where the pre/queries relationships between nodes (i, j) such that i < j and node values in the tree are prejacent to node values. The pre/queens traversal of a binary tree should be as follows: pre1.get(node_id), pre2.get() -> getNode(id).getValue() -> do nothing else. From left to right we have a pre pref
<|endchallenge|>
----------------------------


--- Generated Challenge ---
<|challenge|>
TITLE: Find the Minimum Number of Events That Can Be Attended After Each Event
TYPE: code
LEVEL: advanced
DESCRIPTION: You are given an array events where each event can be scheduled to take place from beginningTime. Note that you should not attend any event without a restriction. Return the minimum number of events 

'<|challenge|>\nTITLE: Find the Number of Nice Substrings That Can Make Two Strings Equal by One\nTYPE: code\nLEVEL: advanced\nDESCRIPTION : Given a string s and a string word, find the number that can make two strings equal by one. Example 1: Input: s = "loveleetcode", word = ["love"] Output: 6 Explanation: "loveleetcode" can be made equal if "loveleetcode" can\'t be made equal by one. The only substring with two different letters is "love". Note: S can have two different letters. It is guaranteed that there will be at least one nice substring in s after each pair(s) is taken into account. Since this question may be too large, return it modulo 109 + 7. A good substring is defined as follows: Any two adjacent characters (possibly zero) that appear exactly once are alike. If no two similar words occur more than once, they do not appear'

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Download model to your PC directly
from google.colab import files
import shutil
import os

# Zip the model folder
shutil.make_archive('challenge_model', 'zip', '.', 'challenge_model')
print("Zipped. Downloading now...")

# Download it
files.download('challenge_model.zip')

Zipped. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from huggingface_hub import HfApi, create_repo

HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
HF_USERNAME = "anascs12"
REPO_NAME   = "buildims-challenge-generator"

api = HfApi()

# Create repo
try:
    create_repo(repo_id=f"{HF_USERNAME}/{REPO_NAME}", token=HF_TOKEN, exist_ok=True)
    print(f"Repo created: {HF_USERNAME}/{REPO_NAME}")
except Exception as e:
    print(f"Repo error: {e}")

# Upload model
print("Uploading model files...")
api.upload_folder(
    folder_path="./challenge_model",
    repo_id=f"{HF_USERNAME}/{REPO_NAME}",
    token=HF_TOKEN,
)
print(f"Done! https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

Repo created: anascs12/buildims-challenge-generator
Uploading model files...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eckpoint-508/optimizer.pt:   1%|1         | 6.82MB /  655MB            

  ...e_model/model.safetensors:   0%|          |  549kB /  328MB            

...

  ...eckpoint-508/scheduler.pt:   6%|6         |  88.0B / 1.47kB            

  ...e_model/training_args.bin:   6%|6         |   311B / 5.14kB            

  ...int-508/training_args.bin:   6%|6         |   311B / 5.14kB            

  ...ckpoint-508/rng_state.pth:   6%|6         |   875B / 14.5kB            

Done! https://huggingface.co/anascs12/buildims-challenge-generator
